In [2]:
## importing the important libraries:
import os
from dotenv import load_dotenv
load_dotenv()

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence.aio import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.ai.documentintelligence.models import DocumentContentFormat
from openai import AsyncAzureOpenAI
import base64


In [3]:
file_path = r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparator\sample\Output 1.pdf"

# Load environment variables
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENT_INTELLIGENCE_KEY")

In [4]:
### function for the reading the pdf as input and return the result
document_intelligence_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

async def data_extractor(pdf_path:str):
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    # Base64 encode PDF
    base64_encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

    analyze_request = {
        "base64Source": base64_encoded_pdf
    }

    # Start analysis
    poller = await document_intelligence_client.begin_analyze_document(
        "prebuilt-layout",
        analyze_request,
        output_content_format=DocumentContentFormat.MARKDOWN
        
    )
    
    result = await poller.result()
    page_wise_md = result.content.split("<!-- PageBreak -->")

    page_wise_ocr = []

    for page_idx, page in enumerate(result.pages):
        page_wise_context = ""
        if not(page.lines):
            import pdb;pdb.set_trace()
        for line_idx, line in enumerate(page.lines):
            page_wise_context += line.content

        page_wise_ocr.append(page_wise_context)

    return page_wise_md, page_wise_ocr


In [5]:
page_wise_md, page_wise_ocr = await data_extractor(file_path)

# Text Cleaning Process:


## Handling the Header and Footer of the Documents:

In [6]:
# for i in range(len(page_wise_ocr)):
#     print(page_wise_ocr[i])

In [76]:
import re

def remove_header_footer_flexible(text):
    """
    More flexible version that can handle variations in the header/footer.
    Handles multiple header formats found in MBR documents.
    """
    
    # Flexible header patterns
    header_patterns = [
        # Main header pattern (page 1 style)
        r'Master Batch Record\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TAB\s*Effective\s*Alembic\s*Touching Lives over["\s]*100\s*years\s*ID/Version/Description\s*F1M00332/00000001/Aripiprazole Tab[\.\s]*USP 5mg',
        
        # Simpler catch-all for header
        r'Master Batch Record\s*30000773[^M]*?Aripiprazole Tab\.?USP 5mg',
        
        # NEW: Repeated header on subsequent pages (Material line)
        r'Master Batch Record\s*Material:\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TAB\s*BO',
        
        # NEW: Variation without "BO" at end
        r'Master Batch Record\s*Material:\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TAB(?=\s*BO|\s*$)',
        
        # NEW: Just the "Master Batch RecordMaterial:" line standalone
        r'Master Batch RecordMaterial:\s*30000773\s*[-–]\s*ARIPIPRAZOLE\s*5\s*MG\s*TABBO',
    ]
    
    # Flexible footer patterns
    footer_patterns = [
        # Standard footer: Page X of Y + Name + Date + Version
        r'Page\s*\d+\s*of\s*\d+\s*[A-Za-z\s]+\d{2}/\d{2}/\d{4}\s*\d{2}:\d{2}:\d{2}\s*V\d+',
        
        # Alternative: Version first
        r'V\d+\s*[A-Za-z\s]+\d{2}/\d{2}/\d{4}\s*\d{2}:\d{2}:\d{2}\s*Page\s*\d+\s*of\s*\d+',
        
        # Just page number pattern
        r'Page\s*\d+\s*of\s*\d+',

        # remove 
        r'group:---________________Verification signature:no',
        r'group:---Verification signature:no________________',
    ]
    
    cleaned_text = text
    
    # Remove all header patterns
    for pattern in header_patterns:
        cleaned_text = re.sub(pattern, ' ', cleaned_text, flags=re.IGNORECASE | re.DOTALL)
    
    # Remove all footer patterns
    for pattern in footer_patterns:
        cleaned_text = re.sub(pattern, ' ', cleaned_text, flags=re.IGNORECASE)
    
    # Clean up whitespace
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    cleaned_text = re.sub(r'[ \t]+', ' ', cleaned_text)
    cleaned_text = cleaned_text.strip(" ")
    
    return cleaned_text


In [77]:
cleaned_text_ocr = [
    remove_header_footer_flexible(text)
    for text in page_wise_ocr
]


In [78]:
len(cleaned_text_ocr)

198

In [79]:
cleaned_text = "\n\n".join(cleaned_text_ocr)

In [81]:
print(cleaned_text)

Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\BMR\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by QADate - User: 10/10/2025 13:20:40 - 11693 / Rahul JangaleEffectiveSet MBR effectiveDate - User: 16/10/2025 15:55:08 - 1558

In [82]:
## store in to txt files
with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\data.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(str(cleaned_text))

## Parent Chunking:

In [ ]:
## parent chunk statergy:
"""
-- Extract the data from the LLM, and OCR
-- Merge the data
-- ID - BO Description: {type}

"""

In [83]:
lst = ['INFO - Batch Information', 'DSPRM - Dispensing RM', 'GRAN - Granulation', 'COMP - Compression', 'INSCOM - Inspection']

In [84]:
empt_abbrevation_lst = []
for i in range(len(lst)):
    text = lst[i]
    text_list = text.split(" - ")
    empt_abbrevation_lst.append(text_list[0])
    
print(empt_abbrevation_lst)

['INFO', 'DSPRM', 'GRAN', 'COMP', 'INSCOM']


{'Approval_1': 'Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by QADate - User: 10/10/2025 13:20:40 - 11693 / Rahul JangaleEffectiveSet MBR effectiveDate - User: 16/10/20

In [ ]:
ALLOWED_PARENTS = [
    "INFO - Batch Information",
    "DSPRM - Dispensing RM",
    "GRAN - Granulation",
    "COMP - Compression",
    "INSCOM - Inspection"
]

## parent keywords need to extracted from the gpt4
PARENT_KEYWORD = "ID - BO Description"

In [85]:
def normalize_text(text: str) -> str:
    """Normalize OCR noise"""
    text = text.replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text)
    return text.strip()


def detect_parent(line: str):
    """
    Detect which allowed parent this line belongs to
    """
    for parent in ALLOWED_PARENTS:
        if parent in line:
            return parent
    return None

In [99]:
from collections import defaultdict

def build_parent_chunks(raw_text: str):
    text = normalize_text(raw_text)
    # print(text)
    lines = text.split("\n")
    print(len(lines))
    
    parent_chunks = defaultdict(list)
    current_parent = "MISCELLANEOUS"

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Detect BO Description
        if PARENT_KEYWORD in line:
            detected = detect_parent(line)
            if detected:
                current_parent = detected
            else:
                current_parent = "MISCELLANEOUS"

        parent_chunks[current_parent].append(line)

    # -------------------------------
    # BUILD FINAL JSON
    # -------------------------------
    output = []
    parent_id = 1

    for parent_name, content_lines in parent_chunks.items():
        merged_text = "\n".join(content_lines).strip()

        output.append({
            "parent_chunk_id": parent_id,
            "parent_chunk_name": parent_name,
            "number_of_child_chunks_possible": len(content_lines),
            "chunk_data": merged_text
        })

        parent_id += 1

    return output


In [100]:
parent_chunks = build_parent_chunks(cleaned_text)

198


In [101]:
parent_chunks

[{'parent_chunk_id': 1,
  'parent_chunk_name': 'MISCELLANEOUS',
  'number_of_child_chunks_possible': 1,
  'chunk_data': 'Standard batch size1,500,000 NOEstimated yield1,500,000 NOLong descriptionProduct Name : Aripiprazole Tablets USP 5 mgGeneric Name of Product : Aripiprazole Tablets USPBrand Name : NAStrength : 5 mgStandard Batch Size in Kg : 142.500 kgStandard Batch Size in Unit : 1,500,000 TabletsReference Document No. : MFC/0353-05Market/Customer : Export /USRef. BMR No. : F1\\BMR\\00837 & 3.0ApprovalStatusTextMaster Batch Record 10#F1M00332#Aripiprazole Tab.USP 5mg#30000773# --- ##00000001DraftObject was createdDate - User: 24/09/2025 15:57:34 - 27759 / Kinjal MehtaIn circulationPut MBR in circulationDate - User: 24/09/2025 16:30:55 - 27759 / Kinjal MehtaIn circulationMBR approval by technical support teamDate - User: 30/09/2025 12:26:34 - 16328 / Yesha PatelIn circulationMBR approval by production teamDate - User: 07/10/2025 14:08:16 - 9183 / Nilesh PatelReleasedMBR approval by 

In [98]:
import json

with open(
    r"C:\Users\SumeetMaheshwari\Desktop\workspace\doc-comparsion-using-Azure-OCR\Backend\artifact\parent_chunking_sop.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(data, f, indent=2, ensure_ascii=False)
